# NeedleTTT: download the three raw datasets (DEE-6)

Run this notebook on a Colab CPU runtime after sharing the team Drive folder and adding its `NeedleTTT` shortcut to My Drive. This downloads **MovieLens-1M**, **Amazon Reviews 2023 Video Games 5-core**, and **Amazon Reviews 2023 Grocery and Gourmet Food 5-core**.

The Amazon ratings and separate product metadata occupy about **2.1 GB** in the shared Drive. They are stored under `data/raw/amazon_video_games/` and `data/raw/amazon_grocery/`. Check source terms before sharing the files with the team. MovieLens is downloaded to the temporary runtime, not the shared Drive; each teammate obtains it directly because [GroupLens does not permit redistribution without separate permission](https://files.grouplens.org/datasets/movielens/ml-1m-README.txt). Its files disappear when the runtime is reset.

Downloads are pinned to a publisher revision, checked against SHA-256 and expected sizes, and reused when already verified. An existing file with different contents is left untouched and reported. This notebook **does not** create model-ready catalogs, user splits, or processed data; those remain DEE-6 work.


In [ ]:
import os
from pathlib import Path
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
else:
    print("Drive is already mounted.")


## Check paths and choose datasets

The default Drive root is `/content/drive/MyDrive/NeedleTTT`. If you use another shortcut location, set `NEEDLETTT_DRIVE_ROOT` in the previous cell or your environment before running the next cell. The notebook refuses to create a missing shared root or missing `data/raw` and `data/processed` folders. Select fewer datasets below if you do not need all three in this run.

For reviewing an unmerged branch, set `NEEDLETTT_DATA_BRANCH` to its branch name before the clone cell. Once the PR is merged, the default `main` branch works.


In [ ]:
DRIVE_ROOT = Path(os.environ.get("NEEDLETTT_DRIVE_ROOT", "/content/drive/MyDrive/NeedleTTT")).expanduser()
DATA_ROOT = DRIVE_ROOT / "data"
LOCAL_ROOT = Path("/content/needlettt-local-data")
DATASETS = ["movielens_1m", "amazon_video_games", "amazon_grocery"]
REPO_BRANCH = os.environ.get("NEEDLETTT_DATA_BRANCH", "main")
REPO_DIR = Path("/content/NeedleTTT-data-setup")

missing = [p for p in (DATA_ROOT / "raw", DATA_ROOT / "processed") if not p.is_dir()]
if missing:
    raise FileNotFoundError("Shared Drive folders are missing: " + ", ".join(map(str, missing)))
print("Shared raw data:", DATA_ROOT / "raw")
print("Local-only MovieLens:", LOCAL_ROOT / "movielens_1m")
print("Selected datasets:", ", ".join(DATASETS))


## Get the download code

The runtime clone is temporary. If a checkout already exists, this notebook checks its branch and prints its commit instead of silently switching or replacing your files. Reset the runtime if you want a fresh clone.


In [ ]:
import subprocess
import sys

REPO_URL = "https://github.com/muyishen2040/NeedleTTT.git"
if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout")
    branch = subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip()
    if branch != REPO_BRANCH:
        raise RuntimeError(f"Existing checkout is on {branch!r}; expected {REPO_BRANCH!r}. Reset the runtime or choose a different path.")
    print("Using existing checkout without updating it:", REPO_DIR)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Dataset code commit:", commit)


## Download and verify the raw files

The first run can take several minutes and needs around 2.1 GB of available shared Drive space. Downloads stream to temporary files and become visible under their final names only after verification. Rerunning checks existing files rather than downloading them again.


In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR / "src") + os.pathsep + env.get("PYTHONPATH", "")
subprocess.run(
    [sys.executable, "-m", "needlettt.data.download", "--data-root", str(DATA_ROOT),
     "--local-root", str(LOCAL_ROOT), "--datasets", *DATASETS],
    env=env,
    check=True,
)


## What comes next

- Amazon raw files are in the two named folders under `data/raw/`; MovieLens is local to this runtime.
- No shared folder is created if it is missing, and no existing mismatched file is overwritten.
- A separate DEE-6 preparation step still needs to produce versioned events, catalogs, train/validation/test splits, filtering counts, and manifests under `data/processed/`.
- Share the code and settings through GitHub; do not commit raw datasets or restricted course code.
